# Movies Project Starter

Use this starter notebook for the BAN 6003 final project option: **The Movies Dataset / MovieLens ratings**.

Run cells from top to bottom first. Then replace starter notes with your own project decisions, checks, and interpretation.

Start each milestone by reading the matching Canvas milestone assignment page. The Canvas page tells you exactly what evidence and submission items are required; this notebook gives you starter spaces to build that work.



## Before You Start: Key Project Hints

This dataset mixes movie-level tables and rating-event-level data. The key skill is controlling granularity.

- A common final ABT is **one row per movie**.
- Do **not** directly merge raw `ratings` into `movies` and assume the result is still one row per movie.
- Aggregate ratings to `movieId` first, then merge the rating summary into the movie table.
- Use `movieId` for ratings and `tmdbId` for curated credits/keywords.
- Average rating is unstable for movies with very few ratings. Use a minimum rating-count threshold before modeling.
- Be careful about leakage: post-release fields such as revenue, popularity, vote count, or rating count may not be valid if your question is pre-release prediction.

## Setup

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists() and (REPO_ROOT.parent / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
REPORTS_DIR = REPO_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)


## Load Data

Read each provided table. Keep the original row meaning in mind before joining tables.

In [2]:
movies = pd.read_csv(DATA_DIR / 'movies_metadata.csv')
ratings = pd.read_csv(DATA_DIR / 'ratings.csv', parse_dates=['rating_datetime'])
links = pd.read_csv(DATA_DIR / 'links.csv')
credits = pd.read_csv(DATA_DIR / 'credits_curated.csv')
keywords = pd.read_csv(DATA_DIR / 'keywords_curated.csv')

tables = {
    'movies': movies,
    'ratings': ratings,
    'links': links,
    'credits': credits,
    'keywords': keywords,
}

## Basic Data Profile

Use this section for your first pass through row counts, columns, data types, missingness, duplicates, and suspicious values.

In [17]:
for name, df in tables.items():
    print(f'\n{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
    display(df.head(3))


movies: 9,082 rows x 21 columns


,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
0,1,862,tt0114709,Toy Story,Toy Story,en,1995-10-30,81.0,30000000,373554033.0,21.946943,7.7,5415.0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Animation|Comedy|Family,Animation,1995
1,2,8844,tt0113497,Jumanji,Jumanji,en,1995-12-15,104.0,65000000,262797249.0,17.015539,6.9,2413.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Adventure|Fantasy|Family,Adventure,1995
2,3,15602,tt0113228,Grumpier Old Men,Grumpier Old Men,en,1995-12-22,101.0,0,0.0,11.712900,6.5,92.0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Romance|Comedy,Romance,1995



ratings: 99,810 rows x 5 columns


,userId,movieId,rating,timestamp,rating_datetime
0,1,31,2.5,1260759144,2009-12-14 02:52:24
1,1,1029,3.0,1260759179,2009-12-14 02:52:59
2,1,1061,3.0,1260759182,2009-12-14 02:53:02



links: 9,125 rows x 3 columns


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0



credits: 9,082 rows x 3 columns


,tmdbId,director,top_cast
0,862,John Lasseter,Tom Hanks|Tim Allen|Don Rickles
1,8844,Joe Johnston,Robin Williams|Jonathan Hyde|Kirsten Dunst
2,15602,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret



keywords: 9,082 rows x 2 columns


,tmdbId,keywords_list
0,862,jealousy|toy|boy|friendship|friends|rivalry|bo...
1,8844,board game|disappearance|based on children's b...
2,15602,fishing|best friend|duringcreditsstinger|old men


In [31]:
profile_rows = []
for name, df in tables.items():
    profile_rows.append({
        'table': name,
        'rows': len(df),
        'columns': df.shape[1],
        'duplicate_rows': int(df.duplicated().sum()),
        'missing_cells': int(df.isna().sum().sum()),
    })
pd.DataFrame(profile_rows)

,table,rows,columns,duplicate_rows,missing_cells
0,movies,9082,21,0,72
1,ratings,99810,5,0,0
2,links,9125,3,0,13
3,credits,9082,3,0,114
4,keywords,9082,2,0,761


### Key and Granularity Audit

Use the expected key for each table before planning joins. `ratings` is the exception: its row meaning is a user-movie rating event, so check the combined `userId`-`movieId` key rather than expecting `movieId` alone to be unique. `links` is a crosswalk; because `movies` already contains both IDs, use it for validation unless your analysis needs one of its fields.


In [44]:
key_audit = pd.DataFrame([
    {"table": "movies", "expected_key": "movieId", "duplicate_keys": movies["movieId"].duplicated().sum(), "missing_key_values": movies["movieId"].isna().sum()},
    {"table": "movies", "expected_key": "tmdbId", "duplicate_keys": movies["tmdbId"].duplicated().sum(), "missing_key_values": movies["tmdbId"].isna().sum()},
    {"table": "ratings", "expected_key": "userId + movieId", "duplicate_keys": ratings[["userId", "movieId"]].duplicated().sum(), "missing_key_values": ratings[["userId", "movieId"]].isna().sum().sum()},
    {"table": "links", "expected_key": "movieId", "duplicate_keys": links["movieId"].duplicated().sum(), "missing_key_values": links["movieId"].isna().sum()},
    {"table": "credits", "expected_key": "tmdbId", "duplicate_keys": credits["tmdbId"].duplicated().sum(), "missing_key_values": credits["tmdbId"].isna().sum()},
    {"table": "keywords", "expected_key": "tmdbId", "duplicate_keys": keywords["tmdbId"].duplicated().sum(), "missing_key_values": keywords["tmdbId"].isna().sum()},
])

key_audit


,table,expected_key,duplicate_keys,missing_key_values
0,movies,movieId,0,0
1,movies,tmdbId,0,0
2,ratings,userId + movieId,0,0
3,links,movieId,0,0
4,credits,tmdbId,0,0
5,keywords,tmdbId,0,0


## Milestone 1 Starter: Initial Profile and Cleaning Plan

Complete this by the first project milestone.

Write notes on:

- Business problem and decision context
- Raw tables and row meaning
- Initial row and column counts
- Data type issues
- Missing values and duplicates
- Suspicious values
- Proposed target/outcome
- Early ethics or governance concerns

Important Movies notes:

- `movies` is movie-level, while `ratings` is user-movie rating-event-level.
- `credits` and `keywords` are curated movie-level tables connected by `tmdbId`.
- Budget and revenue may contain zero or missing-like values; describe this before using them.
- Decide whether your project is about audience ratings, catalog strategy, revenue, popularity, or genre/content patterns.

### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 1 written workspace

Use this cell to draft the narrative required by the Canvas Milestone 1 page.

### Business or Research Question
What factors are associated with higher movie ratings and audience popularity, and how do characteristics such as genre, director, cast, runtime, and release information relate to movie performance?

### Why This Question Matters
This question matters because movie studios and streaming platforms need to understand what types of movies perform well with audiences. Looking at ratings and movie characteristics can help identify patterns that may be useful when making decisions about future content.

### Tables/Files Used
I am using the movies, ratings, links, credits, and keywords tables. The movies table contains the main information about each movie, ratings contains individual user ratings, links connects movie IDs with IMDb and TMDB IDs, credits contains director and cast information, and keywords contains descriptive keywords for each movie.

### Row Meaning of Each Important Table
One row in the movies table represents one movie. One row in the ratings table represents one rating given by a user to a movie. One row in the links table represents the identifiers associated with a movie. One row in the credits table represents the director and top cast information for a movie. One row in the keywords table represents the keywords associated with a movie.

### Early Data Quality Issues
The initial profile shows that the data is generally structured well, but there are some missing values that need attention. The keywords table has the largest missingness among the fields reviewed, with about 8.38% missing in `keywords_list`. The credits table also has some missing values in `top_cast` and `director`, and the links table has a small amount of missing `tmdbId` values. The movies table also has some missing genre information. The initial key audit did not identify duplicate keys or missing key values in the expected keys that were checked.

### Initial Cleaning Plan
I will first review the missing values and decide whether they should be kept, filled, or excluded depending on how each variable will be used. I will also check data types and make sure IDs are consistent before joining tables. After that, I will combine the tables needed for the analysis and verify that the joins do not unexpectedly change the number or meaning of the rows.

### Possible Target/Outcome
A possible outcome is to analyze movie ratings and popularity and identify which movie characteristics are associated with stronger audience performance. The analysis could compare factors such as genre, director, cast, runtime, and other movie information with ratings or popularity.

### Ethics, Privacy, or Governance Concerns
The ratings data contains user IDs, so I should avoid treating those IDs as personal information about specific individuals or trying to identify users. The results should describe patterns in the dataset rather than make assumptions about individual users. I should also document any cleaning or filtering decisions so the analysis remains transparent and reproducible.



In [56]:
# Milestone 1 code workspace
# Add your additional profiling checks here.
# Examples: missingness by important columns, duplicate key checks, suspicious value checks.
print("Movies missing values:")
display(movies.isna().sum().sort_values(ascending=False).head(10))

print("Ratings duplicate user-movie pairs:")
print(ratings.duplicated(subset=["userId", "movieId"]).sum())

print("Movies duplicate movieId:")
print(movies["movieId"].duplicated().sum())

print("Credits missing values:")
display(credits.isna().sum().sort_values(ascending=False))

print("Keywords missing values:")
display(keywords.isna().sum().sort_values(ascending=False))



Movies missing values:


genres_list          35
primary_genre        35
status                2
tmdbId                0
movieId               0
original_title        0
title                 0
imdb_id               0
original_language     0
revenue               0
dtype: int64

Ratings duplicate user-movie pairs:
0
Movies duplicate movieId:
0
Credits missing values:


top_cast    90
director    24
tmdbId       0
dtype: int64

Keywords missing values:


keywords_list    761
tmdbId             0
dtype: int64

In [57]:
# Milestone 1 starter checks
for name, df in tables.items():
    print(f'\n{name}')
    display(df.dtypes.to_frame('dtype').head(20))
    display(df.isna().mean().sort_values(ascending=False).head(10).to_frame('missing_rate'))


movies


,dtype
movieId,int64
tmdbId,int64
imdb_id,str
title,str
original_title,str
original_language,str
release_date,str
runtime,float64
budget,int64
revenue,float64


,missing_rate
genres_list,0.003854
primary_genre,0.003854
status,0.000220
tmdbId,0.000000
movieId,0.000000
original_title,0.000000
title,0.000000
imdb_id,0.000000
original_language,0.000000
revenue,0.000000



ratings


,dtype
userId,int64
movieId,int64
rating,float64
timestamp,int64
rating_datetime,datetime64[us]


...

,dtype
tmdbId,int64
director,str
top_cast,str


,missing_rate
top_cast,0.009910
director,0.002643
tmdbId,0.000000



keywords


,dtype
tmdbId,int64
keywords_list,str


,missing_rate
keywords_list,0.083792
tmdbId,0.000000
